# Advanced Time Series Forecasting, Optimization and Explainability

**Course:** Advanced Topics in Deep Learning — 2nd Semester 2025/2026

**Dataset:** Jena Climate (Weather) — Air Temperature Forecasting

**Task:** Multivariate-input, univariate-output, multi-step forecasting

---

**Group Members:**
- Student 1 — Name (ID)
- Student 2 — Name (ID)
- Student 3 — Name (ID)

## Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Data Loading](#2-data-loading)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Data Preprocessing](#4-data-preprocessing)
5. [Windowing & DataLoaders](#5-windowing--dataloaders)
6. [Baseline Models](#6-baseline-models)
7. [Evolutionary Optimization](#7-evolutionary-optimization)
8. [Synthetic Data Generation (Optional/Bonus)](#8-synthetic-data-generation-optionalbonus)
9. [Explainable AI (XAI)](#9-explainable-ai-xai)
10. [Efficiency & Resource Analysis](#10-efficiency--resource-analysis)
11. [Comparative Analysis & Discussion](#11-comparative-analysis--discussion)
12. [Conclusion](#12-conclusion)

---
## 1. Setup & Imports

Install and import all required libraries. Set random seeds for reproducibility.

In [ ]:
# TODO: Add all required pip installs (uncomment as needed)
# !pip install torch torchvision numpy pandas matplotlib seaborn scikit-learn shap lime tqdm

In [ ]:
# --- Core Libraries ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import warnings
warnings.filterwarnings('ignore')

# --- Deep Learning ---
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# --- Scikit-learn ---
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TODO: Import XAI libraries when needed (Section 9)
# import shap
# from lime import lime_tabular

# TODO: Import evolutionary algorithm libraries when needed (Section 7)
# from deap import base, creator, tools, algorithms  # Option A: DEAP
# import optuna  # Option B: Optuna with evolutionary sampler

# --- Reproducibility ---
SEED = 42

def set_seed(seed=SEED):
    """Set random seeds for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# --- Device Configuration ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## 2. Data Loading

Load the Jena Climate dataset. Keep only the **required variables**:
- `Date Time` — timestamp
- `T (degC)` — air temperature (**TARGET**)
- `p (mbar)` — atmospheric pressure
- `rh (%)` — relative humidity
- `wv (m/s)` — wind speed
- `max. wv (m/s)` — maximum wind speed
- `wd (deg)` — wind direction

In [ ]:
# TODO: Download the Jena Climate dataset if not already present
# Source: https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip
# Steps:
#   1. Download the zip file
#   2. Unzip to a data/ directory
#   3. Load the CSV into a DataFrame

# DATA_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip"
# DATA_DIR = "data/"
# DATA_FILE = os.path.join(DATA_DIR, "jena_climate_2009_2016.csv")

# TODO: Implement data download and loading
# if not os.path.exists(DATA_FILE):
#     os.makedirs(DATA_DIR, exist_ok=True)
#     ... download and unzip ...

# df_raw = pd.read_csv(DATA_FILE)
# print(f"Raw dataset shape: {df_raw.shape}")
# df_raw.head()

In [ ]:
# TODO: Select only the required columns
# REQUIRED_COLS = ['Date Time', 'T (degC)', 'p (mbar)', 'rh (%)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)']
# TARGET_COL = 'T (degC)'
# df = df_raw[REQUIRED_COLS].copy()

# TODO: Parse 'Date Time' as datetime
# df['Date Time'] = pd.to_datetime(df['Date Time'])
# df = df.set_index('Date Time')

# TODO: Check for missing values, duplicates, anomalies
# print(df.isnull().sum())
# print(f"Duplicate timestamps: {df.index.duplicated().sum()}")
# df.describe()

---
## 3. Exploratory Data Analysis

Understand the structure and patterns in the data before modelling.

In [ ]:
# TODO: Plot the target variable (temperature) over time
# - Full time series plot
# - Zoomed-in view (e.g. 1 month) to see daily patterns
# - Comment on visible trends, seasonality, noise

In [ ]:
# TODO: Plot all input variables over time
# - Use subplots, one per variable
# - Look for correlations, patterns, outliers

In [ ]:
# TODO: Correlation analysis
# - Compute and plot the correlation matrix (heatmap)
# - Identify which variables are most correlated with temperature
# - Discuss implications for feature importance

In [ ]:
# TODO: Distribution analysis
# - Histograms or KDE plots for each variable
# - Check for skewness, outliers, or non-standard distributions

---
## 4. Data Preprocessing

Prepare the data for modelling: sub-sampling, temporal covariates, normalization, and splitting.

### 4.1 Sub-sampling

The original data is recorded every 10 minutes. Sub-sample to reduce computational cost.

In [ ]:
# TODO: Sub-sample the data
# - Original: every 10 minutes
# - Suggested: every 1 hour (take every 6th row)
# - Justify the choice of sub-sampling rate

# SUBSAMPLE_RATE = 6  # every 6th row = 1 hour intervals
# df_subsampled = df.iloc[::SUBSAMPLE_RATE].copy()
# print(f"Sub-sampled shape: {df_subsampled.shape}")

### 4.2 Temporal Covariates

Engineer time-based features. Use **cyclic encoding** (sin/cos) for periodic features.

In [ ]:
# TODO: Create temporal covariates from the datetime index
# Suggested features (cyclically encoded where appropriate):
#   - Hour of day  -> sin(2π * hour/24), cos(2π * hour/24)
#   - Day of year  -> sin(2π * day/365), cos(2π * day/365)
#   - Day of week  -> sin(2π * dow/7),   cos(2π * dow/7)
#   - Month        -> sin(2π * month/12), cos(2π * month/12)
#
# IMPORTANT: Justify which temporal covariates you include in the report.
#            Explain WHY cyclic encoding preserves temporal structure.
#
# Example:
# df_subsampled['hour_sin'] = np.sin(2 * np.pi * df_subsampled.index.hour / 24)
# df_subsampled['hour_cos'] = np.cos(2 * np.pi * df_subsampled.index.hour / 24)
# ...

### 4.3 Train / Validation / Test Split

Use a **chronological split** (no shuffling for time series!).

- **Test set must remain UNTOUCHED** until final evaluation.
- Validation set is used for evolutionary fitness evaluation.

In [ ]:
# TODO: Split data chronologically
# Suggested split ratios: 70% train, 15% validation, 15% test
#
# n = len(df_subsampled)
# train_end = int(n * 0.70)
# val_end = int(n * 0.85)
#
# df_train = df_subsampled.iloc[:train_end]
# df_val   = df_subsampled.iloc[train_end:val_end]
# df_test  = df_subsampled.iloc[val_end:]
#
# print(f"Train: {df_train.shape}, Val: {df_val.shape}, Test: {df_test.shape}")

### 4.4 Normalization / Scaling

**IMPORTANT:** Fit the scaler on **training data only** to avoid data leakage.

In [ ]:
# TODO: Normalize the data
# - Choose scaler: MinMaxScaler or StandardScaler (justify choice)
# - Fit on TRAINING data only
# - Transform train, val, and test sets
# - Store the scaler for inverse transformations during evaluation
#
# FEATURE_COLS = [col for col in df_train.columns if col != TARGET_COL]  # or define explicitly
#
# scaler = StandardScaler()  # or MinMaxScaler()
# train_scaled = scaler.fit_transform(df_train)
# val_scaled   = scaler.transform(df_val)
# test_scaled  = scaler.transform(df_test)

---
## 5. Windowing & DataLoaders

Create supervised learning samples using sliding windows:

$$\mathbf{X}_{t-L+1:t} \longrightarrow \mathbf{y}_{t+1:t+H}$$

- **L** = lookback window length (number of past time steps)
- **H** = forecast horizon (number of future time steps to predict)
- **X** = multivariate input (all meteorological variables)
- **y** = univariate output (temperature only)

In [ ]:
# TODO: Define windowing parameters
# These will be optimized by the evolutionary algorithm later (Section 7)
# Start with reasonable defaults from the mini-project

# LOOKBACK = 72   # e.g. 72 hours = 3 days of hourly data
# HORIZON  = 24   # e.g. 24 hours = 1 day forecast
# BATCH_SIZE = 64

In [ ]:
# TODO: Implement the sliding window function
# def create_windows(data, target_idx, lookback, horizon):
#     """
#     Create sliding window samples for multi-step forecasting.
#     
#     Args:
#         data: numpy array of shape (timesteps, features)
#         target_idx: column index of the target variable
#         lookback: L — number of past time steps
#         horizon: H — number of future time steps to predict
#     
#     Returns:
#         X: shape (num_samples, lookback, num_features)
#         y: shape (num_samples, horizon)
#     """
#     X, y = [], []
#     for i in range(lookback, len(data) - horizon + 1):
#         X.append(data[i - lookback:i, :])         # all features
#         y.append(data[i:i + horizon, target_idx])  # target only
#     return np.array(X), np.array(y)

In [ ]:
# TODO: Create windowed datasets for train, val, and test
# target_idx = ...  # index of 'T (degC)' in the feature array
# X_train, y_train = create_windows(train_scaled, target_idx, LOOKBACK, HORIZON)
# X_val,   y_val   = create_windows(val_scaled,   target_idx, LOOKBACK, HORIZON)
# X_test,  y_test  = create_windows(test_scaled,  target_idx, LOOKBACK, HORIZON)
#
# print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
# print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
# print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

In [ ]:
# TODO: Create PyTorch Dataset and DataLoaders
#
# class TimeSeriesDataset(Dataset):
#     def __init__(self, X, y):
#         self.X = torch.tensor(X, dtype=torch.float32)
#         self.y = torch.tensor(y, dtype=torch.float32)
#
#     def __len__(self):
#         return len(self.X)
#
#     def __getitem__(self, idx):
#         return self.X[idx], self.y[idx]
#
# train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
# val_loader   = DataLoader(TimeSeriesDataset(X_val, y_val),     batch_size=BATCH_SIZE, shuffle=False)
# test_loader  = DataLoader(TimeSeriesDataset(X_test, y_test),   batch_size=BATCH_SIZE, shuffle=False)

---
## 6. Baseline Models

Train at least one baseline architecture (GRU or Transformer) from the mini-project.
This serves as the **reference** for all subsequent improvements.

### 6.1 GRU Baseline

In [ ]:
# TODO: Define the GRU model architecture
#
# class GRUForecaster(nn.Module):
#     def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.1, horizon=24):
#         super().__init__()
#         # TODO: Define layers:
#         #   - GRU layer(s) with input_size, hidden_size, num_layers, dropout
#         #   - Fully connected output layer: hidden_size -> horizon
#         #   - Optional: layer norm, residual connections
#
#     def forward(self, x):
#         # TODO: Forward pass:
#         #   1. Pass x through GRU -> output, hidden
#         #   2. Take the last time step's output (or apply attention)
#         #   3. Pass through FC layer
#         #   4. Return predictions of shape (batch_size, horizon)
#         pass

### 6.2 Transformer Baseline

In [ ]:
# TODO: Define Positional Encoding
#
# class PositionalEncoding(nn.Module):
#     def __init__(self, d_model, max_len=5000, dropout=0.1):
#         super().__init__()
#         # TODO: Create sinusoidal positional encoding matrix
#         #   PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
#         #   PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
#         #   Register as buffer (not a learnable parameter)
#
#     def forward(self, x):
#         # TODO: Add positional encoding to input, apply dropout
#         pass

In [ ]:
# TODO: Define the Transformer model architecture
#
# class TransformerForecaster(nn.Module):
#     def __init__(self, input_size, d_model=64, nhead=4, num_layers=2,
#                  dim_feedforward=128, dropout=0.1, horizon=24):
#         super().__init__()
#         # TODO: Define layers:
#         #   - Input projection: Linear(input_size, d_model)
#         #   - Positional encoding
#         #   - TransformerEncoder (with TransformerEncoderLayer)
#         #   - Output FC layer(s)
#
#     def forward(self, x):
#         # TODO: Forward pass:
#         #   1. Project input to d_model dimension
#         #   2. Add positional encoding
#         #   3. Pass through Transformer Encoder
#         #   4. Pool/flatten encoder output
#         #   5. Pass through FC layer
#         #   6. Return predictions of shape (batch_size, horizon)
#         pass

### 6.3 Training Loop & Utilities

In [ ]:
# TODO: Implement the training function with early stopping
#
# def train_model(model, train_loader, val_loader, optimizer, criterion,
#                 num_epochs=100, patience=10, device=DEVICE):
#     """
#     Train a model with early stopping and track efficiency metrics.
#
#     Must track (for Section 5.5 - Efficiency Analysis):
#       - Total training time
#       - Time per epoch
#       - Peak memory usage (GPU if available)
#
#     Returns:
#       - history dict with train/val losses, best epoch, training time
#       - best model state_dict
#     """
#     # TODO: Implement training loop
#     #   - For each epoch: train step, validation step, early stopping check
#     #   - Save best model weights
#     #   - Track timing and memory
#     pass

In [ ]:
# TODO: Implement the evaluation function
#
# def evaluate_model(model, test_loader, criterion, scaler=None, device=DEVICE):
#     """
#     Evaluate a trained model. Compute MSE, RMSE, MAE, MAPE, R².
#     Inverse-transform predictions if scaler is provided.
#     Measure inference time (for Section 5.5).
#     """
#     # TODO: Run inference, collect predictions, compute metrics
#     pass

In [ ]:
# TODO: Utility function to plot training curves
#
# def plot_training_history(history, title="Training History"):
#     # TODO: Plot train loss and val loss vs epoch
#     pass

# TODO: Utility function to plot predictions vs actuals
#
# def plot_predictions(y_true, y_pred, title="Predictions vs Actual"):
#     # TODO: Plot overlay of predicted and actual values
#     pass

### 6.4 Train & Evaluate Baselines

In [ ]:
# TODO: Instantiate and train the GRU baseline
# gru_model = GRUForecaster(input_size=..., hidden_size=64, num_layers=2, horizon=HORIZON).to(DEVICE)
# optimizer = torch.optim.Adam(gru_model.parameters(), lr=1e-3)
# criterion = nn.MSELoss()
#
# gru_history = train_model(gru_model, train_loader, val_loader, optimizer, criterion)
# plot_training_history(gru_history, "GRU Baseline Training")

In [ ]:
# TODO: Evaluate GRU baseline on test set
# gru_results = evaluate_model(gru_model, test_loader, criterion, scaler)
# print(f"GRU Baseline — MSE: {gru_results['mse']:.4f}, MAE: {gru_results['mae']:.4f}")
# plot_predictions(gru_results['y_true'], gru_results['y_pred'], "GRU Baseline Predictions")

In [ ]:
# TODO: Instantiate and train the Transformer baseline
# transformer_model = TransformerForecaster(input_size=..., d_model=64, nhead=4, horizon=HORIZON).to(DEVICE)
# optimizer = torch.optim.Adam(transformer_model.parameters(), lr=1e-3)
#
# transformer_history = train_model(transformer_model, train_loader, val_loader, optimizer, criterion)
# plot_training_history(transformer_history, "Transformer Baseline Training")

In [ ]:
# TODO: Evaluate Transformer baseline on test set
# transformer_results = evaluate_model(transformer_model, test_loader, criterion, scaler)
# print(f"Transformer Baseline — MSE: {transformer_results['mse']:.4f}, MAE: {transformer_results['mae']:.4f}")
# plot_predictions(transformer_results['y_true'], transformer_results['y_pred'], "Transformer Baseline Predictions")

---
## 7. Evolutionary Optimization

Apply **Evolutionary Algorithms** to automatically optimize the end-to-end forecasting pipeline.

This is NOT just hyperparameter tuning — it should optimize the full pipeline including:
- Model hyperparameters (hidden_size, num_layers, dropout, lr, etc.)
- Architectural choices (GRU vs Transformer, bidirectional, attention, etc.)
- Windowing strategies (lookback length, forecast horizon)
- Data preprocessing choices

**Multi-objective optimization** (e.g. error vs. complexity/efficiency) is encouraged.

### 7.1 Genotype Representation

Define what is being optimized and how it is encoded.

In [ ]:
# TODO: Define the search space (genotype representation)
# Each individual in the population represents a full pipeline configuration.
#
# Example search space:
# SEARCH_SPACE = {
#     # --- Windowing ---
#     'lookback':        [24, 48, 72, 96, 120, 168],  # hours
#     'horizon':         [6, 12, 24, 48],              # hours
#
#     # --- Model Architecture ---
#     'model_type':      ['gru', 'transformer'],
#     'hidden_size':     [32, 64, 128, 256],
#     'num_layers':      [1, 2, 3, 4],
#     'dropout':         [0.0, 0.1, 0.2, 0.3, 0.5],
#
#     # --- GRU-specific ---
#     'bidirectional':   [True, False],
#
#     # --- Transformer-specific ---
#     'nhead':           [2, 4, 8],
#     'dim_feedforward': [64, 128, 256],
#
#     # --- Training ---
#     'learning_rate':   [1e-4, 5e-4, 1e-3, 5e-3],
#     'batch_size':      [32, 64, 128],
# }
#
# TODO: Explain the genotype encoding in detail
# TODO: Justify the choice of search ranges

### 7.2 Fitness Function

**IMPORTANT:** Fitness must be computed on the **validation set only** (not test set).

In [ ]:
# TODO: Define the fitness function
# The fitness function decodes a genotype, builds & trains a model, and evaluates on validation set.
#
# def fitness_function(individual):
#     """
#     Evaluate fitness of an individual (pipeline configuration).
#
#     Steps:
#       1. Decode genotype into pipeline configuration
#       2. Create windowed data with the specified lookback/horizon
#       3. Build model with the specified architecture
#       4. Train model (with reduced epochs/budget for speed)
#       5. Evaluate on VALIDATION set
#       6. Return fitness value(s)
#
#     For multi-objective: return (val_loss, num_params) or (val_loss, inference_time)
#     For single-objective: return (val_loss,)
#     """
#     # TODO: Implement fitness evaluation
#     pass
#
# TODO: Consider multi-objective optimization
#   - Objective 1: Minimize validation error (MSE/MAE)
#   - Objective 2: Minimize model complexity (num_params) or inference time
#   - Use Pareto-front analysis if multi-objective

### 7.3 Evolutionary Strategy & Operators

Define selection, crossover, and mutation operators.

In [ ]:
# TODO: Set up the evolutionary algorithm
# Choose a library/framework: DEAP, Optuna (evolutionary sampler), or custom implementation.
#
# Define:
#   - Population size (e.g. 20-50 individuals)
#   - Number of generations (e.g. 10-30)
#   - Selection strategy (tournament, roulette, NSGA-II for multi-objective)
#   - Crossover operator and probability
#   - Mutation operator and probability
#   - Training budget per individual (reduced epochs to keep search feasible)
#
# POPULATION_SIZE = 20
# NUM_GENERATIONS = 15
# CROSSOVER_PROB  = 0.7
# MUTATION_PROB   = 0.2
# TRAINING_BUDGET = 20  # max epochs per individual during search
#
# TODO: Implement the evolutionary loop
# TODO: Log the best fitness per generation
# TODO: Store the Pareto front if multi-objective

### 7.4 Run Evolutionary Search

In [ ]:
# TODO: Run the evolutionary search
# - This is the main optimization loop
# - Track and display progress (best fitness per generation)
# - Save results for analysis

In [ ]:
# TODO: Visualize evolutionary search results
# - Plot fitness evolution over generations
# - If multi-objective: plot Pareto front
# - Show the best individual's configuration

### 7.5 Overfitting Mitigation & Final Evaluation

**IMPORTANT:** Evolutionary search may overfit to the validation set.

In [ ]:
# TODO: Re-evaluate the best individual(s) with multiple random seeds
# - Train the best configuration with 3-5 different random seeds
# - Report mean and std of validation/test performance
# - This addresses overfitting to a single validation split
#
# TODO: Train the final optimized model with full training budget
# - Use the best configuration from evolutionary search
# - Train with full epochs (not reduced budget)
# - Evaluate on the UNTOUCHED test set
#
# TODO: Compare optimized model vs baseline (quantitatively)

---
## 8. Synthetic Data Generation (Optional/Bonus)

Explore generative approaches (e.g., GAN-based) for data augmentation.

**⚠️ Data Leakage Warning:** Synthetic data must be generated using the **training set only**.

In [ ]:
# TODO (OPTIONAL/BONUS): Implement synthetic data generation
#
# Approaches to consider:
#   - TimeGAN (Generative Adversarial Network for time series)
#   - VAE-based generation
#   - Simple augmentation techniques (jittering, scaling, window warping)
#
# Steps:
#   1. Train a generative model on TRAINING data only
#   2. Generate synthetic time series segments
#   3. Evaluate quality of synthetic data (visual inspection, distribution comparison)
#   4. Augment training set with synthetic data
#   5. Re-train forecasting model with augmented data
#   6. Compare performance WITH vs WITHOUT augmentation
#
# IMPORTANT: No data leakage! Only use training data for generation.

---
## 9. Explainable AI (XAI)

Apply XAI techniques to analyse and interpret the forecasting models.

**Required:**
- At least **one global** explanation method
- At least **one local** explanation method

### 9.1 Global Explainability

Understand the model's overall behaviour and feature importance.

In [ ]:
# TODO: Apply at least ONE global explanation method
#
# Options:
#   - Permutation Feature Importance
#   - SHAP summary plots (global feature importance)
#   - Integrated Gradients (averaged over many samples)
#   - Attention weight analysis (for Transformer models)
#
# Steps:
#   1. Choose and justify the method
#   2. Apply to the trained model
#   3. Visualize results (feature importance bar chart, SHAP beeswarm, etc.)
#   4. Interpret and discuss:
#      - Which features are most important for temperature forecasting?
#      - Does this align with domain knowledge?
#      - How does importance change between baseline and optimized models?

### 9.2 Local Explainability

Explain individual predictions — why did the model predict this specific value?

In [ ]:
# TODO: Apply at least ONE local explanation method
#
# Options:
#   - SHAP values for individual predictions
#   - LIME (Local Interpretable Model-agnostic Explanations)
#   - Integrated Gradients (for specific samples)
#   - Attention visualization (for Transformer — which time steps are attended to)
#   - Saliency maps / gradient-based attribution
#
# Steps:
#   1. Choose and justify the method
#   2. Select interesting sample predictions:
#      - A well-predicted sample
#      - A poorly-predicted sample (high error)
#      - An edge case (e.g., sudden temperature change)
#   3. Visualize explanations for each
#   4. Interpret and discuss:
#      - What drove the prediction for each sample?
#      - Which time steps and features mattered most?
#      - Are error cases explainable?

---
## 10. Efficiency & Resource Analysis

Analyse computational costs and trade-offs between performance and efficiency.

**Required metrics:**
- Training time
- Inference time
- Number of model parameters
- Memory usage (RAM and/or GPU)

In [ ]:
# TODO: Collect efficiency metrics for all models
#
# def count_parameters(model):
#     """Count total number of trainable parameters."""
#     return sum(p.numel() for p in model.parameters() if p.requires_grad)
#
# def measure_inference_time(model, test_loader, device, n_runs=10):
#     """Measure average inference time over multiple runs."""
#     # TODO: Time the full inference pass, average over n_runs
#     pass
#
# def measure_memory_usage(model, sample_input, device):
#     """Measure peak GPU/CPU memory during forward pass."""
#     # TODO: Use torch.cuda.max_memory_allocated() or tracemalloc for CPU
#     pass

In [ ]:
# TODO: Create a comprehensive comparison table
#
# Columns: Model | Params | Train Time | Infer Time | Memory | MSE | MAE | R²
# Rows:    GRU Baseline | Transformer Baseline | EA-Optimized | (Augmented)
#
# efficiency_results = pd.DataFrame({
#     'Model': ['GRU Baseline', 'Transformer Baseline', 'EA-Optimized'],
#     'Parameters': [...],
#     'Training Time (s)': [...],
#     'Inference Time (ms)': [...],
#     'Memory (MB)': [...],
#     'MSE': [...],
#     'MAE': [...],
#     'R²': [...]
# })
# efficiency_results

In [ ]:
# TODO: Visualize trade-offs
# - Plot: Accuracy vs Number of Parameters (scatter plot)
# - Plot: Accuracy vs Training Time
# - Plot: Accuracy vs Inference Time
# - Discuss: Is the additional complexity of the optimized model justified?

---
## 11. Comparative Analysis & Discussion

**This is one of the most important parts of the project.**

Provide a thorough analysis covering all aspects of the work.

### 11.1 Baseline vs Optimized Model Comparison

In [ ]:
# TODO: Compare baseline and optimized models
# - Side-by-side prediction plots
# - Metric comparison table (MSE, RMSE, MAE, MAPE, R²)
# - Statistical significance tests if applicable
# - How much did evolutionary optimization improve performance?

### 11.2 Discussion Points

Address each of these in written analysis (markdown cells):

**TODO: Write detailed discussion covering:**

1. **Impact of Evolutionary Optimization**
   - What improvements did the EA achieve over the baseline?
   - What pipeline components had the largest impact?
   - Did multi-objective optimization reveal interesting trade-offs?

2. **Insights from Explainability (XAI)**
   - What did the global explanations reveal about feature importance?
   - What did local explanations show about individual predictions?
   - Do the explanations align with meteorological domain knowledge?
   - How do explanations differ between baseline and optimized models?

3. **Accuracy vs Complexity vs Efficiency Trade-offs**
   - Is the optimized model significantly better than the baseline?
   - Is the improvement worth the additional complexity?
   - How do training/inference costs scale?

4. **Strengths and Limitations of the Final Solution**
   - What works well?
   - What are the failure modes?
   - What would you do differently with more time/resources?
   - Potential improvements and future work.

---
## 12. Conclusion

**TODO: Write a conclusion summarizing:**

- The problem addressed
- Key findings and results
- The most effective pipeline configuration found
- Main insights from explainability analysis
- Efficiency considerations
- Recommendations for future work